# Exploratory Analysis: QSAR for Acetylcholinesterase Inhibitors

This notebook provides an overview of the dataset, feature engineering, and model interpretability for the AChE QSAR project. It is part of a human-AI collaboration experiment between **Semen Gavrilov** and **Manus AI**.

**Author:** Semen Gavrilov
**Date:** 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import joblib
import shap
import os

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Data Overview
We load the curated dataset from ChEMBL for Human Acetylcholinesterase (CHEMBL220).

In [ ]:
train_df = pd.read_csv('data/processed/train.csv')
test_df = pd.read_csv('data/processed/test.csv')
print(f'Training set: {train_df.shape[0]} compounds')
print(f'Test set: {test_df.shape[0]} compounds')
train_df.head()

## 2. Bioactivity Distribution
The target variable is **pIC50** ($-\log_{10}[\text{IC50 M}]$). A higher pIC50 indicates higher potency.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(train_df['pIC50'], kde=True, color='skyblue')
plt.title('Distribution of pIC50 (Training Set)')
plt.xlabel('pIC50')
plt.ylabel('Frequency')
plt.show()

## 3. Chemical Space Visualization
Using Principal Component Analysis (PCA) on the molecular descriptors to visualize the distribution of compounds.

In [ ]:
from sklearn.decomposition import PCA

meta_cols = ['canonical_smiles', 'pIC50', 'active']
X_train = train_df.drop(columns=meta_cols)
y_train = train_df['pIC50']

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)

plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='viridis', alpha=0.6)
plt.colorbar(label='pIC50')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('Chemical Space Visualization (PCA)')
plt.show()

## 4. Model Interpretability (SHAP)
We use SHAP to understand which features drive the XGBoost model's predictions.

In [ ]:
reg_model = joblib.load('models/xgboost_regressor.joblib')
selected_features = joblib.load('models/selected_features.joblib')
X_test = test_df[selected_features]

explainer = shap.TreeExplainer(reg_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Top Features by SHAP Importance')
plt.show()

## 5. Sample Active Molecules
Visualizing some of the most potent inhibitors in the training set.

In [ ]:
top_mols = train_df.nlargest(8, 'pIC50')
mols = [Chem.MolFromSmiles(s) for s in top_mols['canonical_smiles']]
Draw.MolsToGridImage(mols, legends=[f'pIC50: {p:.2f}' for p in top_mols['pIC50']], molsPerRow=4)